# Amostra 5 x 3 - Satélite, Segmentado e Infravermelho

Este notebook encontra cinco IDs presentes nas três pastas e mostra uma linha por amostra.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from PIL import Image

BASE = Path('/Users/arthurvtl/IFES/IC')
PASTA_SATELITE = BASE / 'ORIGINAL'
PASTA_SEGMENTADO = BASE / 'SEGMENTADO'
PASTA_INFRAVERMELHO = BASE / 'INFRAVERMELHO' / 'NIR'
QUANTIDADE = 5
ARQUIVO_SAIDA = Path('artifacts/amostra_5x3.png')

print('Satélite:', PASTA_SATELITE)
print('Segmentado:', PASTA_SEGMENTADO)
print('Infravermelho:', PASTA_INFRAVERMELHO)

## Encontrar IDs comuns

O pareamento é feito pelo número em `amostra_N`, evitando colocar imóveis diferentes na mesma linha.

In [ ]:
def indexar_imagens(pasta: Path, padrao: str) -> dict[int, Path]:
    if not pasta.is_dir():
        raise FileNotFoundError(f'Pasta não encontrada: {pasta}')
    regex = re.compile(padrao, re.IGNORECASE)
    encontrados = {}
    for caminho in pasta.iterdir():
        if not caminho.is_file():
            continue
        resultado = regex.fullmatch(caminho.name)
        if resultado:
            encontrados[int(resultado.group(1))] = caminho
    return encontrados

satelites = indexar_imagens(
    PASTA_SATELITE,
    r'amostra_(\d+)_satelite_1920\.(?:tif|tiff|png)',
)
segmentados = indexar_imagens(
    PASTA_SEGMENTADO,
    r'amostra_(\d+)_uso_solo_1920\.(?:tif|tiff|png)',
)
infravermelhos = indexar_imagens(
    PASTA_INFRAVERMELHO,
    r'amostra_(\d+)_cbers_nir\.(?:tif|tiff)',
)

ids_comuns = sorted(set(satelites) & set(segmentados) & set(infravermelhos))
if len(ids_comuns) < QUANTIDADE:
    raise RuntimeError(
        f'Existem apenas {len(ids_comuns)} IDs completos; são necessários {QUANTIDADE}.'
    )

ids_selecionados = ids_comuns[:QUANTIDADE]
print('IDs selecionados:', ids_selecionados)

## Ler e preparar as imagens

RGB é exibido nas cores originais. NIR recebe apenas contraste visual por percentis; o GeoTIFF original não é alterado.

In [ ]:
def ler_rgb(caminho: Path) -> np.ndarray:
    if caminho.suffix.lower() in {'.png', '.jpg', '.jpeg'}:
        return np.asarray(Image.open(caminho).convert('RGB'))
    with rasterio.open(caminho) as src:
        if src.count < 3:
            raise ValueError(f'Imagem RGB tem menos de três bandas: {caminho}')
        imagem = src.read([1, 2, 3]).transpose(1, 2, 0)
    if imagem.dtype == np.uint8:
        return imagem
    saida = np.zeros(imagem.shape, dtype=np.float32)
    for banda in range(3):
        valores = imagem[..., banda].astype(np.float32)
        validos = valores[np.isfinite(valores)]
        minimo, maximo = np.percentile(validos, [2, 98])
        if maximo > minimo:
            saida[..., banda] = np.clip((valores - minimo) / (maximo - minimo), 0, 1)
    return saida

def ler_nir(caminho: Path):
    with rasterio.open(caminho) as src:
        imagem = src.read(1, masked=True).astype(np.float32)
    validos = imagem.compressed()
    if validos.size == 0:
        raise ValueError(f'NIR sem pixels válidos: {caminho}')
    minimo, maximo = np.percentile(validos, [2, 98])
    return imagem, float(minimo), float(maximo)

## Grade final 5 x 3

In [ ]:
figura, eixos = plt.subplots(
    QUANTIDADE,
    3,
    figsize=(15, 4.4 * QUANTIDADE),
    constrained_layout=True,
)

titulos = ['Satélite RGB', 'Segmentado', 'Infravermelho NIR']
for coluna, titulo in enumerate(titulos):
    eixos[0, coluna].set_title(titulo, fontsize=14, fontweight='bold')

for linha, amostra_id in enumerate(ids_selecionados):
    eixos[linha, 0].imshow(ler_rgb(satelites[amostra_id]))
    eixos[linha, 1].imshow(ler_rgb(segmentados[amostra_id]))
    nir, minimo, maximo = ler_nir(infravermelhos[amostra_id])
    eixos[linha, 2].imshow(nir, cmap='gray', vmin=minimo, vmax=maximo)
    eixos[linha, 0].set_ylabel(
        f'Amostra {amostra_id}',
        fontsize=12,
        fontweight='bold',
    )
    for coluna in range(3):
        eixos[linha, coluna].set_xticks([])
        eixos[linha, coluna].set_yticks([])

figura.suptitle('Comparação de cinco amostras', fontsize=18, fontweight='bold')
ARQUIVO_SAIDA.parent.mkdir(parents=True, exist_ok=True)
figura.savefig(ARQUIVO_SAIDA, dpi=170, bbox_inches='tight')
print('Figura salva em:', ARQUIVO_SAIDA.resolve())
plt.show()

A figura também fica salva em `artifacts/amostra_5x3.png`.